# Checkpoint 4 — Clean the Data

**Goal:** Take the raw data and convert it into a form the ML model can actually use.

**What we'll do (in order):**
1. Remove the useless ID column
2. Fix the TotalCharges column (text → number)
3. Convert Yes/No columns → 1/0
4. Convert multi-choice columns → numbers (one-hot encoding)
5. Save the clean file so Phase 2 can use it

**Rule:** Run one cell at a time. Read the output. Understand what changed.

In [1]:
import pandas as pd

# Load the original raw file — we never modify this file directly
df = pd.read_csv("../data/raw/telco_churn.csv")

print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nAll columns:")
print(list(df.columns))

Loaded: 7043 rows, 21 columns

All columns:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


## Step 1 — Remove the customerID column

`customerID` is just a random code like "7590-VHVEG". 
It has ZERO relationship to whether a customer churns.
If we leave it in, the model might try to "learn" from it — which would be nonsense.

**Rule:** Always remove ID columns before training a model.

In [2]:
# Drop the customerID column
# axis=1 means "drop a column" (axis=0 would mean drop a row)
df = df.drop(columns=["customerID"])

print(f"Columns now: {df.shape[1]}  (was 21, now 20 — customerID removed)")
print("\nRemaining columns:")
print(list(df.columns))

Columns now: 20  (was 21, now 20 — customerID removed)

Remaining columns:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


## Step 2 — Fix TotalCharges (text → number, remove 11 blank rows)

We found in Checkpoint 2 that `TotalCharges` is stored as text and has 11 hidden blank rows.

Two things to fix:
- Convert the whole column from text to a number
- Remove the 11 rows that are blank (they become NaN after conversion)

In [3]:
# Convert TotalCharges from text to number
# errors="coerce" → if a value can't be converted (like blank " "), make it NaN
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print(f"NaN values in TotalCharges: {df['TotalCharges'].isna().sum()}  ← these are the 11 blank rows")

# Remove those 11 rows with NaN
df = df.dropna(subset=["TotalCharges"])

print(f"\nRows after removing blanks: {df.shape[0]}  (was 7043, now 7032)")
print(f"TotalCharges data type is now: {df['TotalCharges'].dtype}  ← float64 means decimal number ✅")

NaN values in TotalCharges: 11  ← these are the 11 blank rows

Rows after removing blanks: 7032  (was 7043, now 7032)
TotalCharges data type is now: float64  ← float64 means decimal number ✅


## Step 3 — Convert Yes/No columns to 1/0

Many columns contain only two values: "Yes" or "No".
Models can't read words — so we replace them:
- "Yes" → 1
- "No"  → 0

Same for `gender`: "Male" → 1, "Female" → 0

In [4]:
# These columns only have "Yes" or "No" values
yes_no_columns = [
    "Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"
]

# Replace "Yes" with 1 and "No" with 0
for col in yes_no_columns:
    df[col] = df[col].map({"Yes": 1, "No": 0})

# gender has "Male" / "Female"
df["gender"] = df["gender"].map({"Male": 1, "Female": 0})

print("After conversion — sample values:")
print(df[["gender", "Partner", "Churn"]].head(5))

After conversion — sample values:
   gender  Partner  Churn
0       0        1      0
1       1        0      0
2       1        0      1
3       1        0      0
4       0        0      1


## Step 4 — One-Hot Encoding for multi-choice columns

Some columns have MORE than 2 choices. For example `Contract` has 3:
- "Month-to-month"
- "One year"
- "Two year"

We CAN'T just do: Month-to-month=1, One year=2, Two year=3.
That would tell the model "Two year is 3x bigger than Month-to-month" — which is nonsense.

Instead we create **one new column per choice**, each containing just 0 or 1:

| Contract_Month-to-month | Contract_One year | Contract_Two year |
|---|---|---|
| 1 | 0 | 0 |
| 0 | 1 | 0 |
| 0 | 0 | 1 |

This is called **one-hot encoding**. `pd.get_dummies()` does it automatically.

In [5]:
# These columns have 3+ choices — they need one-hot encoding
multi_choice_columns = [
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaymentMethod"
]

# get_dummies() creates one column per unique value
# drop_first=True removes one column per group to avoid redundancy
# (if Contract_Month-to-month=0 AND Contract_One-year=0, it must be Two-year — so we don't need all 3)
df = pd.get_dummies(df, columns=multi_choice_columns, drop_first=True)

print(f"Columns after one-hot encoding: {df.shape[1]}")
print("\nAll columns now:")
print(list(df.columns))

Columns after one-hot encoding: 31

All columns now:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'Churn', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


## Step 5 — Save the clean file

We save the cleaned data to `data/processed/` folder.

Why a separate folder?
- `data/raw/` = original file, NEVER touched
- `data/processed/` = clean, model-ready file

This is standard practice. If something goes wrong, you always have the original.

In [7]:
import os

# Create the processed folder if it doesn't exist
os.makedirs("../data/processed", exist_ok=True)

# Save the clean dataframe as a new CSV
output_path = "../data/processed/telco_churn_clean.csv"
df.to_csv(output_path, index=False)

print(f"Clean file saved to: {output_path}")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nAll columns are now numbers — ready for ML ✅")
print(f"\nData types:")
print(df.dtypes.value_counts())

Clean file saved to: ../data/processed/telco_churn_clean.csv
Shape: 7032 rows, 31 columns

All columns are now numbers — ready for ML ✅

Data types:
bool       21
int64       8
float64     2
Name: count, dtype: int64


## Your Observations — Fill This In

**Q1: How many columns did we start with? How many do we have now? Why did the number go up?**
21 to 31 , increased because of one hot encoding

**Q2: What was wrong with TotalCharges and how did we fix it?**
11 blank rows were deleted

**Q3: Why can't we just leave "Yes"/"No" as text for the model?**
because ML model understands numbers , not texts

**Q4: What is one-hot encoding? Explain it in your own words.**
we encode any text or words into numbers

**Q5: Why do we save to data/processed/ instead of overwriting the raw file?**
because if something goes wrong , the original data will be available for re-training with other model

---
Paste your answers in the chat when done.
After this, we push everything to GitHub and then start Phase 2 — building the actual ML model.